# Foutanalyse en toepassingsgrenzen

Gebruik buitenfoldvoorspellingen voor de foutanalyse. Subgroepen beschrijven samenhang; ze bewijzen geen oorzaak en hebben soms weinig records.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
HERE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'results_v3/run.json').exists())
sys.path.insert(0, str(HERE))
from mushroom_alternative import development_data, scores
from improve_v3 import experiments, InputFeatures, WEIGHTS
from build_v3_reports import label
X, y, ids, folds = development_data()
summary = pd.read_csv(HERE/'results_v3/summary.csv').set_index('model')
details = pd.read_csv(HERE/'results_v3/outer_fold_metrics.csv')
predictions = pd.read_csv(HERE/'results_v3/outer_oof_predictions.csv', float_precision='round_trip')
selections = json.loads((HERE/'results_v3/selections.json').read_text())
assert len(summary) == 14 and summary.folds.eq(5).all()


In [2]:
decision = json.loads((HERE/'results_v3/decision.json').read_text())
name = decision['best_goal_qualifying_procedure'] or 'rf_compatible_f1_threshold'
p = predictions[predictions.model==name].set_index('source_row').loc[ids].copy()
p['missing_count'] = X.isna().sum(axis=1).to_numpy()
p['missing_group'] = pd.cut(p.missing_count, [-1,2,4,11], labels=['0–2','3–4','5–11'])
rows = []
for group, part in p.groupby('missing_group', observed=True):
    row = {'groep': str(group), 'rijen':len(part), 'giftig':int(part.true_class.sum())}
    if part.true_class.nunique()==2:
        row.update(scores(part.true_class,part.probability_p,prediction=part.prediction))
    rows.append(row)
print('Procedure:', label(name))
display(pd.DataFrame(rows))

Procedure: Gewogen voting + F1-drempel


,groep,rijen,giftig,accuracy,macro_f1,balanced_accuracy,recall_p,precision_p,specificity_e,roc_auc,average_precision,tn,fp,fn,tp
0,0–2,1221,480,0.846028,0.837023,0.833886,0.777083,0.821586,0.890688,0.866529,0.823571,660,81,107,373
1,3–4,2326,876,0.778590,0.758229,0.752822,0.648402,0.732903,0.857241,0.821556,0.760226,1243,207,308,568
2,5–11,553,196,0.763110,0.727462,0.718737,0.566327,0.707006,0.871148,0.785914,0.717691,311,46,85,111


## Hoe verschillend zijn de fouten?

De kruistabel laat zien welke fouten een nieuwe procedure herstelt en welke ze toevoegt. Betere gemiddelde prestaties maken een individueel geval niet betrouwbaar.

In [3]:
baseline=predictions[predictions.model=='rf_compatible_f1_threshold'].set_index('source_row').loc[ids]
display(pd.crosstab(baseline.prediction.ne(baseline.true_class),p.prediction.ne(p.true_class),rownames=['v2 fout'],colnames=['gekozen procedure fout']))
print('Nog gemiste giftige gevallen:', int(((p.true_class==1)&(p.prediction==0)).sum()))

Nog gemiste giftige gevallen: 500


gekozen procedure fout,False,True
v2 fout,,
False,3116,113
True,150,721


## Controle van de onderzoeksketen

De tests reconstrueren metrics vanuit opgeslagen individuele voorspellingen, controleren originele rijlidmaatschappen en reconstrueren votingkansen uit de gekozen gewichten. Export- en API-controles staan afzonderlijk bij de kandidaat.

In [4]:
for name in ['test_results.json','package_test_results.json','frontend_test_results.json']:
    receipt=json.loads((HERE/'results_v3'/name).read_text())
    assert receipt['success']
    print(name)
    display({k:v for k,v in receipt.items() if k not in ['output','hashes']})

test_results.json
package_test_results.json
frontend_test_results.json


{'finished_at_utc': '2026-10-15T13:53:45.846730+00:00',
 'tests_run': 8,
 'failures': 0,
 'errors': 0,
 'skipped': 0,
 'success': True,
 'scope': 'Feature preparation, original row membership, recomputed metrics and voting equivalence. No production integration.',
 'hash_mode': 'LF-normalized SHA256 for text; raw binary hashes'}

{'finished_at_utc': '2026-10-15T14:05:38.145670+00:00',
 'tests_run': 12,
 'failures': 0,
 'errors': 0,
 'skipped': 0,
 'success': True,
 'scope': 'RF/voting export and actual API/form contract, plus all seven v2 regression checks; no production/Citi Bike integration.',
 'hash_mode': 'LF-normalized SHA256 for text; raw binary hashes'}

{'finished_at_utc': '2026-10-15T14:07:18.505985+00:00',
 'success': True,
 'tests_run': 9,
 'exit_code': 0,
 'test_sha256_lf': '44d50e38b891cf91fd138e7f60feda6544b0229ff639d5aa19a84d2a9f9d700b',
 'scope': 'Actual generated v2/v3 frontend decision functions, boundaries and form metadata; not a browser/server run.'}

Geen eindtest of eetbaarheidsbevestiging. Controleer bij latere integratie artifact, voorbereiding, drempel, metrics, versiecontract en automatische training samen. De gezamenlijke applicatie is hier niet aangepast.